In [ ]:
# conda activate genomic_tools

import pandas as pd
from pyfaidx import Fasta

Goal: narrow down the space of exons being considered for Interproscan

In [ ]:
signif_exons = pd.read_csv("data/ctype_exons/annotated/All_GABAergic_exons_annotated.csv")

signif_exons = signif_exons.rename(columns={signif_exons.columns[0]: "event"})

In [ ]:
def score_transcript(x):
    score = 0
    if "MANE_Select" in str(x['tag']):
        score += 1
    if "appris_principal" in str(x['tag']):
        score += 1
    if "basic" in str(x['tag']):
        score += 1
    if "CCDS" in str(x['tag']):
        score += 1
    if "GENCODE_Primary" in str(x['tag']):
        score += 1
    if x['overlap_type'] == "fully_coding":
        score += 1
    return score

In [ ]:
signif_coding_exons = signif_exons[signif_exons['transcript_type'] == "protein_coding"]
signif_coding_exons['transcript_score'] = signif_coding_exons.apply(score_transcript, axis=1)

idx = signif_coding_exons.groupby("event")['transcript_score'].idxmax()
signif_coding_exons_max = signif_coding_exons.loc[idx]

## Structured Domains (longer, structural units)

**InterProScan — the gold standard; searches Pfam, PRINTS, PANTHER, Gene3D, SUPERFAMILY, CDD, and more in one run. Highly recommended for scale.**

In [ ]:
proteins = Fasta("/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v49.pc_translations.fa")

In [ ]:
# Build a dict keyed by ENST
protein_by_transcript = {}
for key in proteins.keys():
    parts = key.split("|")
    enst_versioned = parts[1].split(".")[0]
    protein_by_transcript[enst_versioned] = str(proteins[key])

In [ ]:
transcripts_of_interest = list(set(signif_exons['transcript']))

In [ ]:
modified_transcript_products = {}
transcript_log = []

with open("data/proteins.fa", "w") as f:
    for key in proteins.keys():
        enst = key.split("|")[1].split(".")[0]
        if enst in transcripts_of_interest:
            transcript_log.append(enst)
            seq = str(proteins[key]).rstrip("*")
            if "X" in seq:
                # Track where in the sequence the X was 
                modified_transcript_products[enst] = [i for i, c in enumerate(seq) if c == "X"]
                seq = seq.replace("X", "")
            f.write(f">{enst}\n{seq}\n")